In [3]:
import rasterio
from rasterio.warp import transform
import numpy as np
from scipy.interpolate import griddata
import xarray as xr

In [1]:
def get_weather_for_satellite_extent(satellite_tif_path, weather_ds, date, 
                                      vars_to_use=['tmax', 'tmin', 'prcp', 'srad']):
    """
    Satellite 이미지가 커버하는 지역의 weather만 추출
    
    Returns:
        (224, 224, num_vars) - satellite와 정확히 같은 지역
    """
    # 1. Satellite의 각 픽셀 좌표 (lat/lon) 구하기
    with rasterio.open(satellite_tif_path) as src:
        rows, cols = np.meshgrid(np.arange(224), np.arange(224), indexing='ij')
        xs, ys = rasterio.transform.xy(src.transform, rows.flatten(), cols.flatten())
        xs = np.array(xs).reshape(224, 224)
        ys = np.array(ys).reshape(224, 224)
        
        # 투영좌표 -> lat/lon 변환
        lons, lats = transform(src.crs, 'EPSG:4326', xs.flatten(), ys.flatten())
        sat_lats = np.array(lats).reshape(224, 224)
        sat_lons = np.array(lons).reshape(224, 224)
    
    # 2. 해당 날짜의 weather 데이터
    weather_day = weather_ds.sel(time=date)
    
    # Weather의 lat/lon
    weather_lat = weather_day['lat'].values.flatten()
    weather_lon = weather_day['lon'].values.flatten()
    
    # Satellite 픽셀 좌표들
    sat_points = np.column_stack([sat_lons.flatten(), sat_lats.flatten()])
    
    # 3. 각 변수마다 Satellite 좌표에 맞춰 interpolation
    features = []
    for var in vars_to_use:
        weather_var = weather_day[var].values.flatten()
        
        # NaN 제거
        valid_mask = ~np.isnan(weather_var)
        weather_points = np.column_stack([weather_lon[valid_mask], 
                                          weather_lat[valid_mask]])
        weather_values = weather_var[valid_mask]
        
        # Satellite 픽셀 위치에서의 weather 값 interpolation
        matched_1d = griddata(
            weather_points,   # weather 격자점 좌표
            weather_values,   # weather 값
            sat_points,       # satellite 픽셀 좌표 (224x224개)
            method='nearest'  # 또는 'linear'
        )
        
        matched_2d = matched_1d.reshape(224, 224)
        matched_2d = np.nan_to_num(matched_2d, nan=0.0)
        features.append(matched_2d)
    
    # (224, 224, num_vars)
    return np.stack(features, axis=-1)



In [4]:
weather_ds = xr.open_dataset('/work/mech-ai-scratch/rtali/gis-weather/final_processed_weather/daymet_iowa_2023.nc')
satellite_path = '/work/mech-ai-scratch/bgekim/project/imputation/IA_dataset/30m/Patches/Elevation/patch_9968_9968/elevation_IA.tif'

weather_aligned = get_weather_for_satellite_extent(
    satellite_path,
    weather_ds,
    '2023-09-04',
    vars_to_use=['tmax', 'tmin', 'prcp', 'srad']
)

# print(f"Shape: {weather_aligned.shape}")  # (224, 224, 4)
# print(f"Satellite와 정확히 같은 6.7km x 6.7km 영역의 weather")

In [5]:
import xarray as xr

ds = xr.open_dataset('/work/mech-ai-scratch/rtali/gis-weather/final_processed_weather/daymet_iowa_2023.nc')

# 모든 변수 확인
print(ds)
print("\nAvailable variables:")
print(list(ds.data_vars))

<xarray.Dataset> Size: 2GB
Dimensions:                  (x: 539, y: 371, time: 365)
Coordinates:
  * x                        (x) float64 4kB 2.598e+05 2.608e+05 ... 7.978e+05
  * y                        (y) float64 3kB 1.5e+05 1.49e+05 ... -2.2e+05
  * time                     (time) datetime64[ns] 3kB 2023-01-01 ... 2023-12-31
Data variables:
    lambert_conformal_conic  int64 8B ...
    dayl                     (time, y, x) float32 292MB ...
    lat                      (y, x) float64 2MB ...
    lon                      (y, x) float64 2MB ...
    prcp                     (time, y, x) float32 292MB ...
    srad                     (time, y, x) float32 292MB ...
    swe                      (time, y, x) float32 292MB ...
    tmax                     (time, y, x) float32 292MB ...
    tmin                     (time, y, x) float32 292MB ...
    vp                       (time, y, x) float32 292MB ...
Attributes:
    citation:            Please see http://daymet.ornl.gov/ for current Da